 # Excercise regression
# Author:Haoyang Qian
# Date:2026/09/01 


## Local Linear Regression

This model estimates mortality rates (`MORT`) based on poverty levels (`POOR`) at specified target points ($x_0$). For each target point, it selects the $k$-nearest neighbors, computes distance-based linear kernel weights ($w_i = 1 - d_i / d_{\text{total}}$), and fits a Weighted Least Squares (WLS) regression line. It outputs local predictions ($\hat{y}$) along with their corresponding locally weighted Standard Errors ($SE$).

In [19]:
import numpy as np
import matplotlib as plt
import csv

In [40]:
def find_nearest_points(x:np.array,y:np.array,k:int,x0:np.array):
    se=[]
    pred=[]
    for xa in x0:
        d_all = np.abs(x - xa)
        d_sorted = np.sort(d_all)
        d_thres = d_sorted[k - 1]

        mask = np.abs(x - xa) <= d_thres #find x with less k distance 
        x_nearest = x[mask]
        y_nearest = y[mask]
        d_nearest = d_all[mask]

        d_total = np.sum(d_nearest)
        w = 1 - (d_nearest / d_total)

        sum_w = np.sum(w)
        x_bar_w = np.sum(w * x_nearest) / sum_w
        y_bar_w = np.sum(w * y_nearest) / sum_w

        denom = np.sum(w * (x_nearest - x_bar_w) ** 2)
        beta_1 = np.sum(w * (x_nearest - x_bar_w) * (y_nearest - y_bar_w)) / denom
        beta_0 = y_bar_w - beta_1 * x_bar_w

        pred = np.append(pred, beta_0 + beta_1 * xa)

        y_nearest_fit = beta_0 + beta_1 * x_nearest
        residuals = y_nearest - y_nearest_fit
        se_a = np.sqrt(np.sum(w * (residuals ** 2)) / sum_w)
        se = np.append(se, se_a)
    
    return pred,se
        

In [41]:
pollution_data = np.genfromtxt('pollution_cleaneddata.csv', delimiter=',', skip_header=1)

In [42]:
poor=pollution_data[:,10]
mort=pollution_data[:,15]
res=find_nearest_points(poor,mort,k=5,x0=[10,18,25])

In [43]:
print(f'Under the poverty level of 10,18,25, estimated mortality is{np.round(res[0],2)} with standard error{np.round(res[1],2)}')

Under the poverty level of 10,18,25, estimated mortality is[ 885.03  968.77 1025.78] with standard error[42.06 39.13 49.41]
